# Typ rekord (record)

## Úvod a motivácia

### Príklad - trieda Person

Máme definovanú nasledovnú triedu a dva objekty s rovnakými hodnotami:

In [ ]:
class Person 
{
    public string FirstName { get; }
    public string LastName { get; }

    public Person(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }
}

var person1 = new Person("Jan", "Mrkvicka");
var person2 = new Person("Jan", "Mrkvicka");

Aké hodnoty vrátia nasledujúce výrazy?

In [ ]:
person1.Equals(person2)

In [ ]:
object.Equals(person1, person2)

In [ ]:
person1 == person2

In [ ]:
person1.ToString()

In [ ]:
Console.WriteLine(person2)

Ako musíme upraviť triedu, aby prvé tri výrazy vracali hodnotu *True* a posledné dve vrátili, resp. vypísali reťazec v tvare *"Person { FirstName = Jan, LastName = Mrkvicka }"*?

### Príklad - upravená trieda Person

In [ ]:
class Person
{
    public string FirstName { get; }
    public string LastName { get; }

    public Person(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }

    public override bool Equals(object obj)
    {
        return obj is Person person &&
               FirstName == person.FirstName &&
               LastName == person.LastName;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(FirstName, LastName);
    }

    public static bool operator ==(Person left, Person right)
    {
        return EqualityComparer<Person>.Default.Equals(left, right);
    }

    public static bool operator !=(Person left, Person right)
    {
        return !(left == right);
    }

    public override string ToString()
    {
        return $"Person {{ FirstName = {FirstName}, LastName = {LastName} }}";
        //return $"{nameof(Person)} {{ {nameof(FirstName)} = {FirstName}, {nameof(LastName)} = {LastName} }}"; 
    }
}

var person1 = new Person("Jan", "Mrkvicka");
var person2 = new Person("Jan", "Mrkvicka");

Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(person1 == person2);
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

Celkom dlhá trieda. Môžeme ju trošku zjednodušiť a skrátiť?

### Príklad - upravená a zjednodušená trieda Person

In [ ]:
class Person
{
    public string FirstName { get; }
    public string LastName { get; }

    public Person(string firstName, string lastName) => (FirstName, LastName) = (firstName, lastName);

    public override bool Equals(object obj) => obj is Person person &&
               FirstName == person.FirstName &&
               LastName == person.LastName;

    public override int GetHashCode() => HashCode.Combine(FirstName, LastName);

    public static bool operator ==(Person left, Person right) => EqualityComparer<Person>.Default.Equals(left, right);

    public static bool operator !=(Person left, Person right) => !(left == right);

    public override string ToString() => $"{nameof(Person)} {{ {nameof(FirstName)} = {FirstName}, {nameof(LastName)} = {LastName} }}";
}

var person1 = new Person("Jan", "Mrkvicka");
var person2 = new Person("Jan", "Mrkvicka");

Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(person1 == person2);
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

Stále je celkom dlhá. Neexistuje niečo lepšie?

### Príklad - štruktúra Person

Ak by sme nepotrebovali operátor ==, môžeme deklarovať Person ako štruktúru:

In [ ]:
struct Person
{
    public string FirstName { get; }
    public string LastName { get; }

    public Person(string firstName, string lastName) => (FirstName, LastName) = (firstName, lastName);

    //public static bool operator ==(Person left, Person right) => EqualityComparer<Person>.Default.Equals(left, right);

    //public static bool operator !=(Person left, Person right) => !(left == right);

    public override string ToString() => $"{nameof(Person)} {{ {nameof(FirstName)} = {FirstName}, {nameof(LastName)} = {LastName} }}";
}

var person1 = new Person("Jan", "Mrkvicka");
var person2 = new Person("Jan", "Mrkvicka");

Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
//Console.WriteLine(person1 == person2);
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

Neexistuje predsa niečo lepšie?

### Príklad - rekord Person

Môžeme si definovať pozičný **rekord**:

In [ ]:
record Person(string FirstName, string LastName);

var person1 = new Person("Jan", "Mrkvicka");
var person2 = new Person("Jan", "Mrkvicka");

Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(person1 == person2);
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

## Čo je to rekord?

Typ **record** je *referenčný* typ - špecializovaná trieda, pre ktorú kompilátor vygeneruje rôzne syntetizované metódy:
- pre zistovanie rovností na základe hodnôt (musia sedieť typy a hodnoty všetkých vlastnosti a hodnôt),
    - `override bool Equals(object)`
    - `virtual bool Equals(T)` - implementácia rozhrania `System.IEquatable<T>`, kde parametrom je typ rekordu
    - `operator==`, `operator!=`
- pre zisťovanie hashu - `override int GetHashCode()`
- pre zobrazenie výstupu vo forme reťazca - `override string ToString()`, pričom táto metóda vracia reťazec vyprodukovaný z metódy `PrintMembers(StringBuilder)`, ktorú je možné vytvoriť aj vlastnú 
- pre kopírovanie a klonovanie 

Ak sa jedná o **pozičný rekord** (positional record), kompilátor navyše vygeneruje:
- **primárny konštruktor**, ktorého parametre sú rovnaké ako pozičné parametre v deklarácii rekordu,
- **verejné inicializačné** (public init-only) **vlastnosti** pre každý parameter primárneho konštruktora (tieto init vlastnosti je možné nastaviť iba v konštruktore alebo pomocou inicializácie vlastnosti),
- **metódu** `Deconstruct()` na extrakciu hodnôt vlastností z rekordu. Táto metóda sa vygeneruje iba, ak sú definované 2 alebo viac pozičných parametrov, pričom ignoruje vlastnosti, ktoré sú definované syntaxou so štandardými vlastnosťami. 

Kompilátor môže vygenerovať aj ďalšie metódy podľa potreby. Ak si definujete vlastnú metódu, ktorú kompilátor štandardne generuje ako syntetickú metódu, kompilátor ju nevygeneruje.

Rekord môźeme vytvoriť pomocou:
1. syntaxe s **pozičnými parametrami** (positional parameters),
2. syntaxe so **štandardnýmmi vlastnosťami** (standard properties),
3. kombináciou.

Typ bol zavedený v C# 9.0.

## Príklady rekordov

Najjednoduchší spôsob vytvorenia rekordu je pomocou **syntaxe s pozičnými parametrami**:

In [ ]:
public record Person(string FirstName, string LastName, DateTime Birthdate);

var person = new Person("Ján", "Mrkvička", new DateTime(2000, 12, 31));
person

Iný spôsob je pomocou **syntaxe so štandardnými vlastnosťami** (pozor - konštruktor je v tomto prípade iba bezparametrický): 

In [ ]:
public record Person 
{
    public string FirstName { get; init; }
    public string LastName { get; init; }
    public DateTime Birthdate { get; init; }    
};

// Error CS1729: 'Person' does not contain a constructor that takes 3 arguments
//var person = new Person("Ján", "Mrkvička", new DateTime(2000, 12, 31));

// Môžeme nastaviť jednotlivé vlastnosti v inicializačnej časti:
var person = new Person() { FirstName = "Ján", LastName = "Mrkvička", Birthdate = new DateTime(2000, 12, 31) };
display(person);

// Ak nenastavíme, všetky vlastnosti budú mať implicitné (default) hodnoty
var person2 = new Person();
display(person2);


Môžeme použiť aj **kombináciu pozičných parametrov s vlastnosťami**:

In [ ]:
public record Person(string FirstName, string LastName) 
{
    public DateTime Birthdate { get; init; }
}

// Error CS1729: 'Person' does not contain a constructor that takes 3 arguments
//var person = new Person("Ján", "Mrkvička", new DateTime(2000, 12, 31));

var person = new Person("Ján", "Mrkvička");
display(person);

var person2 = new Person("Eleonóra", "Mrkvičková") { Birthdate = new DateTime(2000, 12, 31) };
display(person2);

Ak potrebujeme, môžeme si definovať aj **konštruktory**:

In [ ]:
public record Person 
{
    public string FirstName { get; init; }
    public string LastName { get; init; }
    public DateTime Birthdate { get; init; }    

    public Person(string firstName, string lastName) => (FirstName, LastName) = (firstName, lastName);
    public Person(string firstName, string lastName, DateTime birthdate) => (FirstName, LastName, Birthdate) = (firstName, lastName, birthdate);
}

// Error CS1729: 'Person' does not contain a constructor that takes 0 arguments
//var person = new Person();

var person = new Person("Ján", "Mrkvička");
display(person);

var person2 = new Person("Eleonóra", "Mrkvičková", new DateTime(2000, 12, 31));
display(person2);

Môžeme kombinovať, pridať aj **ďalšie vlasnosti** a **konštruktor**:

In [1]:
public record Person(string FirstName, string LastName, DateTime Birthdate) 
{
    public string FullName => $"{FirstName} {LastName}";

    public int Age 
    { 
        get 
        {
            var today = DateTime.Today;
            var age = today.Year - Birthdate.Year;
            if (Birthdate.Date > today.AddYears(-age)) 
                age--;
            return age;
        } 
    }

    public Person(string firstName, string lastName) : this(firstName, lastName, new DateTime(1900, 1, 1)) {}
}

var person = new Person("Ján", "Mrkvička");
display(person);

var person2 = new Person("Eleonóra", "Mrkvičková", new DateTime(2000, 12, 31));
display(person2);

var person3 = new Person("Fero", "Nový") { Birthdate = DateTime.Now.AddYears(-30) };
display(person3);

FirstName,LastName,Birthdate,FullName,Age
Ján,Mrkvička,1900-01-01 00:00:00Z,Ján Mrkvička,123


FirstName,LastName,Birthdate,FullName,Age
Eleonóra,Mrkvičková,2000-12-31 00:00:00Z,Eleonóra Mrkvičková,22


FirstName,LastName,Birthdate,FullName,Age
Fero,Nový,1993-03-28 22:53:45Z,Fero Nový,30


Ak nám nevyhovuje **modifikátor prístupu** vlastnosti (dokonca aj pozičného parametra), môžeme si ho zmeniť (napr. na `internal`):
```csharp
    public record Person(string FirstName, string LastName)
    {
        internal string FirstName { get; init; } = FirstName;
    }
```
Pozor, tu je nutné ešte nastaviť vlastnosť na hodnotu získanú z pozičného parametra (`FirstName`). Ak by ste neuviedli žiadny modifikátor prístupu, vlastnosť bude implicitne `public`. 

Otázky:
- Ak zmeníme modifikátor prístupu na `protected`, bude možné skompilovať kód?
- Ak zmeníme modifikátor prístupu na `protected internal`, bude možné skompilovať kód?
- Ak odstránime inicializáciu vlastnosti `FirstName`, čo bude v tejto vlastnosti?  

In [6]:
public record Person(string FirstName, string LastName)
{
    internal string FirstName { get; init; } = FirstName;
}

var person = new Person("Ján", "Mrkvička");
display(person);
Console.WriteLine($"{person.FirstName} {person.LastName}")

LastName,FirstName
Mrkvička,Ján


Ján Mrkvička


Implicitne (defaultne) sú rekordy **nemenné (immutable)**, preto nie je možné priradiť hodnotu do init vlastnosti (zobrazí chybu):

In [7]:
public record Person(string FirstName, string LastName);

// Error CS8852: Init-only property or indexer 'Person.FirstName' can only be assigned in an object initializer, or on 'this' or 'base' in an instance constructor or an 'init' accessor.
var person = new Person("Ján", "Mrkvička");
person.FirstName = "Fero";

Error: (5,1): error CS8852: Init-only property or indexer 'Person.FirstName' can only be assigned in an object initializer, or on 'this' or 'base' in an instance constructor or an 'init' accessor.

Ak by sme ale definovali **vlastnosť** s časťou **set** (namiesto init), potom vlastnosť budeme môcť zmeniť: 

In [8]:
public record Person(string FirstName, string LastName) 
{
    public string LastName { get; set; } = LastName;
}

var person = new Person("Eleonóra", "Slobodná");
display(person);

person.LastName = "Vydatá";
display(person);

FirstName,LastName
Eleonóra,Slobodná


FirstName,LastName
Eleonóra,Vydatá


Pozičné vlastnosti sú síce nemenné, avšak **údaje v** ich **referenčných hodnotách**, na ktoré odkazujú, **je možné meniť** - napríklad pole je referenčný typ, ktorého údaje je možné modifikovať: 

In [9]:
public record Person(string FirstName, string LastName, string[] PhoneNumbers);

var person = new Person("Ján", "Mrkvička", new[] { "+421 944 123 456" });
Console.WriteLine(person.PhoneNumbers[0]);

// Zmeniť celé pole nie je možné, pretože vlastnosť PhoneNumbers je definovaná ako init (nie set)
// Error CS8852: Init-only property or indexer 'Person.PhoneNumbers' can only be assigned in an object initializer, or on 'this' or 'base' in an instance constructor or an 'init' accessor.
//person.PhoneNumbers = new[] { "+421 944 123 456" };

// Nastaviť hodnotu v poli je možné:
person.PhoneNumbers[0] = "+421 908 666 666";
Console.WriteLine(person.PhoneNumbers[0]);

+421 944 123 456
+421 908 666 666


Pozrime sa na metódy pre rovnosť dvoch inštancií:

In [10]:
public record Person(string FirstName, string LastName, string[] PhoneNumbers);

var phoneNumbers = new string[2];
var person1 = new Person("Ján", "Mrkvička", phoneNumbers);
var person2 = new Person("Ján", "Mrkvička", phoneNumbers);

display(person1);
display(person2);

Console.WriteLine(person1 == person2);
Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(object.ReferenceEquals(person1, person2));
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);


FirstName,LastName,PhoneNumbers
Ján,Mrkvička,"[ <null>, <null> ]"


FirstName,LastName,PhoneNumbers
Ján,Mrkvička,"[ <null>, <null> ]"


True
True
True
False
Person { FirstName = Ján, LastName = Mrkvička, PhoneNumbers = System.String[] }
Person { FirstName = Ján, LastName = Mrkvička, PhoneNumbers = System.String[] }


Keď **zmeníme telefónne číslo prvej osoby**, budú sa osoby rovnať?

In [11]:
person1.PhoneNumbers[0] = "+421 908 666 666";

display(person1);
display(person2);

Console.WriteLine(person1 == person2);
Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(object.ReferenceEquals(person1, person2));
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

FirstName,LastName,PhoneNumbers
Ján,Mrkvička,"[ +421 908 666 666, <null> ]"


FirstName,LastName,PhoneNumbers
Ján,Mrkvička,"[ +421 908 666 666, <null> ]"


True
True
True
False
Person { FirstName = Ján, LastName = Mrkvička, PhoneNumbers = System.String[] }
Person { FirstName = Ján, LastName = Mrkvička, PhoneNumbers = System.String[] }


Ak by sme chceli **porovnať dva rekordy s inými názvami rekordov**, ale s rovnakým počtom a typmi pozičných parametrov, budú sa objekty rovnať?

In [12]:
public record Person(string FirstName, string LastName);
public record Osoba(string FirstName, string LastName);

var person1 = new Osoba("Ján", "Mrkvička");
var person2 = new Person("Ján", "Mrkvička");

// Error CS0019: Operator '==' cannot be applied to operands of type 'Osoba' and 'Person'
//Console.WriteLine(person1 == person2);
Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(object.ReferenceEquals(person1, person2));
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

False
False
False
Osoba { FirstName = Ján, LastName = Mrkvička }
Person { FirstName = Ján, LastName = Mrkvička }


Ak potrebujete **zmeniť nemeniteľnú (immutable) vlastnosť** inštancie rekordu, je možné použiť výraz `with` a dosiahnuť tzv. **nedeštruktívnu mutáciu** (nondestructive mutation): 

In [17]:
public record Person(string FirstName, string LastName) 
{
    public string[] PhoneNumbers { get; init; }
}

var person1 = new Person("Jana", "Mrkvičková") { PhoneNumbers = new[] { "+421 944 123 456" } };
var person2 = person1 with { LastName = "Vydatá" };

display(person1);
display(person2);

Console.WriteLine(person1 == person2);
Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(object.ReferenceEquals(person1, person2));
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

FirstName,LastName,PhoneNumbers
Jana,Mrkvičková,[ +421 944 123 456 ]


FirstName,LastName,PhoneNumbers
Jana,Vydatá,[ +421 944 123 456 ]


False
False
False
False
Person { FirstName = Jana, LastName = Mrkvičková, PhoneNumbers = System.String[] }
Person { FirstName = Jana, LastName = Vydatá, PhoneNumbers = System.String[] }


Výraz `with` vytvorí novú inštanciu rekordu, ktorá je kópiou existujúcej inštancie rekordu, pričom sa vytvára tzv. **plytká kópia** (*shallow copy*). To znamená, že sa v prípade referenčných typov vlastností, kopírujú iba ich referencie na inštancie (nie celé objekty). Za výrazom `with` sa špecifikuje zoznam vlastností a fieldov, ktoré sa majú zmodifikovať `{ Property1 = Value, Property2 = Value, ... }`). Vlastnosti musia mať `init` alebo `set` prístupy.

Ak **zmeníme meno** opäť a ešte aj **telefónne číslo druhej osoby**, dostaneme:

In [23]:
person2 = person2 with { LastName = "Kalerábiková" };
person2.PhoneNumbers[0] = null;

display(person1);
display(person2);

Console.WriteLine(person1 == person2);
Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(object.ReferenceEquals(person1, person2));
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

FirstName,LastName,PhoneNumbers
Jana,Mrkvičková,[ <null> ]


FirstName,LastName,PhoneNumbers
Jana,Kalerábiková,[ <null> ]


False
False
False
False
Person { FirstName = Jana, LastName = Mrkvičková, PhoneNumbers = System.String[] }
Person { FirstName = Jana, LastName = Kalerábiková, PhoneNumbers = System.String[] }


Môžeme dokonca vytvoriť **plytkú kópiu prvej osoby** bez zmeny:

In [24]:
person2 = person1 with { };

display(person1);
display(person2);

Console.WriteLine(person1 == person2);
Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(object.ReferenceEquals(person1, person2)); // False
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

FirstName,LastName,PhoneNumbers
Jana,Mrkvičková,[ <null> ]


FirstName,LastName,PhoneNumbers
Jana,Mrkvičková,[ <null> ]


True
True
True
False
Person { FirstName = Jana, LastName = Mrkvičková, PhoneNumbers = System.String[] }
Person { FirstName = Jana, LastName = Mrkvičková, PhoneNumbers = System.String[] }


Ak priradíme do premennej hodnotu inej inštancie, dochádza ku **kópii referencie na tú istú (jednu) inštanciu** rekordu:

In [25]:
person2 = person1;

display(person1);
display(person2);

Console.WriteLine(person1 == person2);
Console.WriteLine(person1.Equals(person2));
Console.WriteLine(object.Equals(person1, person2));
Console.WriteLine(object.ReferenceEquals(person1, person2)); // True
Console.WriteLine(person1.ToString());
Console.WriteLine(person2);

FirstName,LastName,PhoneNumbers
Jana,Mrkvičková,[ <null> ]


FirstName,LastName,PhoneNumbers
Jana,Mrkvičková,[ <null> ]


True
True
True
True
Person { FirstName = Jana, LastName = Mrkvičková, PhoneNumbers = System.String[] }
Person { FirstName = Jana, LastName = Mrkvičková, PhoneNumbers = System.String[] }


## Príklady - dedičnosť

Rekord môžeme podediť pomocou dvojbodky s uvedením triedy a špecifikovaním hodnôt primárneho konštruktora základnej triedy:

In [26]:
public abstract record Person(string FirstName, string LastName);

public record Student(string FirstName, string LastName, int Grade) : Person(FirstName, LastName);

Person student = new Student("Ján", "Mrkvička", 2);
display(student);
Console.WriteLine(student);

Grade,FirstName,LastName
2,Ján,Mrkvička


Student { FirstName = Ján, LastName = Mrkvička, Grade = 2 }


Ak ešte vytvoríme učiteľa, ktorý dedí od osoby, budú sa inštancie rovnať?



In [27]:
public record Teacher(string FirstName, string LastName, int Grade) : Person(FirstName, LastName);

Person teacher = new Teacher("Ján", "Mrkvička", 2);
Person student = new Student("Ján", "Mrkvička", 2);
Console.WriteLine(teacher);
Console.WriteLine(student);
Console.WriteLine(teacher == student);

Console.WriteLine();

Student student2 = new Student("Ján", "Mrkvička", 2);
Console.WriteLine(student2);
Console.WriteLine(student);
Console.WriteLine(student2 == student);


Teacher { FirstName = Ján, LastName = Mrkvička, Grade = 2 }
Student { FirstName = Ján, LastName = Mrkvička, Grade = 2 }
False

Student { FirstName = Ján, LastName = Mrkvička, Grade = 2 }
Student { FirstName = Ján, LastName = Mrkvička, Grade = 2 }
True


## Príklady - Deconstruct metóda

Kompilátor generuje metódu `Deconstruct()` pre pozičné rekordy. Vďake nej môžeme dekonštruovať n-ticu (tuple) tak, že do nej priradíme objekt študenta (`(first, last, number) = student`): 


In [29]:
public record Student(string FirstName, string LastName, int Grade);

var student = new Student("Ján", "Mrkvička", 2);
string first, last;
int number;
(first, last, number) = student;
Console.WriteLine($"{first} {last} {number}");

Ján Mrkvička 2


Môžeme to tiež zjednodušiť s použitím kľúčového slova var:

In [28]:
public record Student(string FirstName, string LastName, int Grade);

var student = new Student("Ján", "Mrkvička", 2);
var (first, last, number) = student;
Console.WriteLine($"{first} {last} {number}");

Ján Mrkvička 2


Ak nejakú vlastnosť nepotrebujeme, môžeme použiť `_` (*discard*) premennú:

In [ ]:
public record Student(string FirstName, string LastName, int Grade);

var student = new Student("Ján", "Mrkvička", 2);
var (first, last, _) = student;
Console.WriteLine($"{first} {last}");

## Zdroje

- [Records (C# reference)](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/builtin-types/record)
- [Introduction to record types in C#](https://learn.microsoft.com/en-us/dotnet/csharp/fundamentals/types/records)
- [What's new in C# 9.0 | Record types](https://learn.microsoft.com/en-us/dotnet/csharp/whats-new/csharp-9#record-types)
- [Tutorials - Explore record types](https://learn.microsoft.com/en-us/dotnet/csharp/whats-new/tutorials/records)
- [Mads Torgersen: C# 9.0 on the record](https://devblogs.microsoft.com/dotnet/c-9-0-on-the-record/)